## Modelado.


La variable objetivo del modelo es log_ratio, definida como el logaritmo del ratio de viajeros, lo que permite estabilizar la varianza y mejorar el ajuste de los modelos.

In [3]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import json
from sklearn.model_selection import TimeSeriesSplit

In [5]:
train_viajeros = pd.read_csv("train_viajeros.csv")
df_verano_viajeros = pd.read_csv("verano_viajeros.csv")
df_viajeros_totales = pd.read_csv("viajeros_totales.csv")

Se quita el mes de mayo del conjunto de entrenamiento. Solo se usa para crear la variable lag_1 para predecir el mes de junio.

In [6]:
train_viajeros = train_viajeros[train_viajeros["Mes"].isin([6,7,8])].copy()

In [7]:
train_viajeros = (train_viajeros.sort_values(["Año", "Mes"]).reset_index(drop=True))

Definición de variables.

In [8]:
x_columnas = ["Provincias", "Año", "Mes", "log_lag_estacional", "log_lag_1", "log_lag_2", "log_lag_3", "log_lag_4", "media_lag_2", "media_lag_3", "tendencia_corta"] 
y_columnas = "log_ratio"

categoricas_viajeros = [x_columnas.index("Provincias")]

### Modelo base: Naive

In [9]:
datos_2024 = train_viajeros[train_viajeros["Año"] == 2024]
true_naive = datos_2024[y_columnas]

prediccion_naive = np.zeros(len(true_naive))

mae_naive = mean_absolute_error(true_naive, prediccion_naive)
rmse_naive = np.sqrt(mean_squared_error(true_naive, prediccion_naive))

print("--- MODELO NAIVE ---")
print("MAE:", mae_naive)
print("RMSE:", rmse_naive)

--- MODELO NAIVE ---
MAE: 0.165972487826781
RMSE: 0.22103603168146027


## Algoritmos candidatos

### 1. CatBoost

Selección de hiperparámetros: Grid Search con TimeSeriesSplit.

In [10]:

tscv = TimeSeriesSplit(n_splits=3)

depth_values = [4, 5, 6]
learning_rates = [0.02, 0.03, 0.04]
l2_values = [3, 4, 5, 6]

resultados_viajeros = []

for depth in depth_values:
    for learning in learning_rates:
        for l2 in l2_values:

            cv_rmses = []

            cv_maes = []

            for train_index, validacion_index in tscv.split(train_viajeros):

                x_train_fold = train_viajeros.iloc[train_index][x_columnas]
                y_train_fold = train_viajeros.iloc[train_index][y_columnas]

                x_validacion_fold = train_viajeros.iloc[validacion_index][x_columnas]
                y_validacion_fold = train_viajeros.iloc[validacion_index][y_columnas]

                modelo = CatBoostRegressor(
                    iterations = 700,
                    learning_rate = learning,
                    depth = depth,
                    l2_leaf_reg = l2,
                    loss_function = "RMSE",
                    random_seed = 42,
                    early_stopping_rounds = 50,
                    verbose=0
                )

                modelo.fit(x_train_fold, y_train_fold, cat_features=categoricas_viajeros, eval_set=(x_validacion_fold, y_validacion_fold))

                prediccion = modelo.predict(x_validacion_fold)

                cv_rmses.append(np.sqrt(mean_squared_error(y_validacion_fold, prediccion)))
                cv_maes.append(mean_absolute_error(y_validacion_fold, prediccion))

            avg_rmse = np.mean(cv_rmses)
            avg_mae = np.mean(cv_maes)
            
            print(f"Depth: {depth}, Learning_rate: {learning}, l2: {l2}")

            resultados_viajeros.append({"depth" : depth, "learning_rate" : learning, "l2" : l2, "MAE" : avg_mae, "RMSE" : avg_rmse})


Depth: 4, Learning_rate: 0.02, l2: 3
Depth: 4, Learning_rate: 0.02, l2: 4
Depth: 4, Learning_rate: 0.02, l2: 5
Depth: 4, Learning_rate: 0.02, l2: 6
Depth: 4, Learning_rate: 0.03, l2: 3
Depth: 4, Learning_rate: 0.03, l2: 4
Depth: 4, Learning_rate: 0.03, l2: 5
Depth: 4, Learning_rate: 0.03, l2: 6
Depth: 4, Learning_rate: 0.04, l2: 3
Depth: 4, Learning_rate: 0.04, l2: 4
Depth: 4, Learning_rate: 0.04, l2: 5
Depth: 4, Learning_rate: 0.04, l2: 6
Depth: 5, Learning_rate: 0.02, l2: 3
Depth: 5, Learning_rate: 0.02, l2: 4
Depth: 5, Learning_rate: 0.02, l2: 5
Depth: 5, Learning_rate: 0.02, l2: 6
Depth: 5, Learning_rate: 0.03, l2: 3
Depth: 5, Learning_rate: 0.03, l2: 4
Depth: 5, Learning_rate: 0.03, l2: 5
Depth: 5, Learning_rate: 0.03, l2: 6
Depth: 5, Learning_rate: 0.04, l2: 3
Depth: 5, Learning_rate: 0.04, l2: 4
Depth: 5, Learning_rate: 0.04, l2: 5
Depth: 5, Learning_rate: 0.04, l2: 6
Depth: 6, Learning_rate: 0.02, l2: 3
Depth: 6, Learning_rate: 0.02, l2: 4
Depth: 6, Learning_rate: 0.02, l2: 5
D

Selección de los mejores parámetros.

In [14]:
df_resultados_viajeros = pd.DataFrame(resultados_viajeros)
df_resultados_viajeros = df_resultados_viajeros.sort_values("RMSE")

mejores_parametros = df_resultados_viajeros.iloc[0]
mejor_rmse = mejores_parametros["RMSE"]

diferecnia = rmse_naive - mejores_parametros["RMSE"]

if mejores_parametros["RMSE"] < rmse_naive:
    print(f"El modelo mejora al Naive")
else:
    print("El modelo no mejora al Naive")

parametros = {"depth" : int(mejores_parametros["depth"]), "learning_rate" : float(mejores_parametros["learning_rate"]), "l2" : int(mejores_parametros["l2"])}

print("Mejores parámetros:")
print(mejores_parametros)

El modelo mejora al Naive
Mejores parámetros:
depth            6.000000
learning_rate    0.030000
l2               3.000000
MAE              0.117266
RMSE             0.157509
Name: 28, dtype: float64


In [15]:
archivo = "parámetros_columnas.json"

datos = {
    "mejores_parametros" : parametros,
    "x_columnas" : x_columnas,
    "y_columnas" : y_columnas
}

with open (archivo, "w") as f:
    json.dump(datos, f, indent=4)


### Otros modelos candidatos (comparativa)

En esta sección comparamos varios algoritmos usando el ismo esquema de validación temporal (`TimeSeriesSplit`) y las mismas métricas (MAE y RMSE) Para modelos que no aceptan variables categóricas directamente, aplicamos One-Hot Encoding solo sobre `Provincias`.


### 2. XGBoost Regressor

In [20]:
from xgboost import XGBRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [21]:
X = train_viajeros[x_columnas]
y = train_viajeros[y_columnas]

Preprocesado (One-Hot solo para Provincias)

In [22]:
preprocess = ColumnTransformer(
    transformers=[
        ("prov", OneHotEncoder(handle_unknown="ignore"), ["Provincias"])
    ],
    remainder="passthrough"
)


Pipeline con XGBoost

In [23]:
xgb_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=600,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

pipe_xgb = Pipeline(
    steps=[
        ("prep", preprocess),
        ("model", xgb_model)
    ]
)


Validación temporal (misma que CatBoost)

In [24]:
tscv = TimeSeriesSplit(n_splits=3)

rmse_cv = []
mae_cv = []

for train_idx, val_idx in tscv.split(X):
    X_train_fold = X.iloc[train_idx]
    y_train_fold = y.iloc[train_idx]
    X_val_fold = X.iloc[val_idx]
    y_val_fold = y.iloc[val_idx]

    pipe_xgb.fit(X_train_fold, y_train_fold)
    preds = pipe_xgb.predict(X_val_fold)

    rmse_cv.append(np.sqrt(mean_squared_error(y_val_fold, preds)))
    mae_cv.append(mean_absolute_error(y_val_fold, preds))

rmse_xgb = np.mean(rmse_cv)
mae_xgb = np.mean(mae_cv)

print("=== XGBoost CV ===")
print("MAE:", mae_xgb)
print("RMSE:", rmse_xgb)


=== XGBoost CV ===
MAE: 0.11841872711022178
RMSE: 0.15977110088774546


## COMPARATIVAS

### Comparación directa de XGBoost Regressor  con CatBoost y Naive

In [25]:
comparativa = pd.DataFrame([
    {"Modelo": "Naive", "MAE": mae_naive, "RMSE": rmse_naive},
    {"Modelo": "CatBoost", "MAE": mejores_parametros["MAE"], "RMSE": mejores_parametros["RMSE"]},
    {"Modelo": "XGBoost", "MAE": mae_xgb, "RMSE": rmse_xgb}
]).sort_values("RMSE")

comparativa


,Modelo,MAE,RMSE
1,CatBoost,0.117266,0.157509
2,XGBoost,0.118419,0.159771
0,Naive,0.165972,0.221036


Conclusiones parciales:

Por ahora, CatBoost es el mejor modelo en ambas métricas.
XGBoost queda muy cerca.Ambos modelos mejoran claramente al modelo Naive.
La diferencia entre CatBoost y XGBoost, aunque pequeña, es consistente.

Por otro lado, CatBoost trabaja directamente con Provincias, XGBoost necesita One-Hot Encoding, lo cual implcica mayor complejidad y dimensionalidad